# Aula 11 - Protocolo Agente-para-Agente (A2A)


## Configuração


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## O que é o Protocolo A2A?

O **protocolo Agente-para-Agente (A2A)** é um padrão aberto que permite que agentes de IA se comuniquem,
se descubram e colaborem — mesmo quando são construídos em frameworks diferentes ou hospedados
por serviços distintos.

Conceitos-chave:

- **Descoberta** – Os agentes publicam um *Agent Card* que descreve suas capacidades, facilitando
  que outros agentes (ou orquestradores) encontrem o especialista certo para uma tarefa.
- **Troca de Mensagens** – Os agentes trocam mensagens estruturadas através de um protocolo comum, para que
  uma solicitação de um agente possa ser entendida e atendida por outro, independentemente de sua
  implementação interna.
- **Ciclo de Vida da Tarefa** – O A2A define estados como *enviada*, *em andamento*, *concluída* e
  *falhou*, dando ao orquestrador visão completa sobre o progresso de uma tarefa delegada.

Nesta lição simulamos uma colaboração no estilo A2A conectando três agentes de viagem especializados
em um fluxo de trabalho onde cada agente contribui com sua expertise e passa os resultados para o próximo.


## Criando Agentes de Viagem Especializados


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Colaboração Multiagente via Fluxo de Trabalho

Conectamos os três agentes em um fluxo de trabalho sequencial que espelha a passagem de mensagens A2A:

1. **CurrencyExchangeAgent** recebe a solicitação do usuário e produz orientação sobre moeda.
2. **ActivityPlannerAgent** recebe o contexto enriquecido e adiciona recomendações de atividades.
3. **TravelManagerAgent** sintetiza ambas as entradas em um resumo final de viagem.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Entendendo o A2A em Produção

Em um ambiente de produção, o protocolo A2A desbloqueia cenários poderosos entre serviços:

| Capacidade | Descrição |
|---|---|
| **Interoperabilidade entre frameworks** | Um agente construído com um framework pode delegar tarefas para um agente construído com qualquer outro framework compatível com A2A, permitindo verdadeira interoperabilidade entre organizações. |
| **Limites de serviço** | Agentes podem residir em microsserviços separados, regiões na nuvem ou até mesmo em organizações diferentes, enquanto colaboram perfeitamente. |
| **Descoberta dinâmica** | Um orquestrador pode consultar um registro de Agent Cards em tempo de execução para encontrar o especialista mais adequado para uma sub-tarefa. |
| **Streaming e notificações push** | O A2A suporta Server-Sent Events (SSE) para atualizações em tempo real e notificações push para tarefas de longa duração. |

O fluxo de trabalho que construímos acima é uma versão simplificada, em processo, desse padrão. Em uma implantação real
cada agente exporia um endpoint HTTP, publicaria um Agent Card e se comunicaria
via protocolo A2A JSON-RPC.


## Summary

In this lesson you learned:

1. **What the A2A protocol is** — an open standard for agent-to-agent discovery, messaging,
   and task management.
2. **How to create specialized agents** — a Currency Exchange agent, an Activity Planner agent,
   and a Travel Manager orchestrator.
3. **How to wire agents into a workflow** — using `WorkflowBuilder` to model sequential
   message passing between agents.
4. **How A2A works in production** — enabling cross-framework, cross-service collaboration
   with dynamic discovery and streaming updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
